# Compare all of Parameters:
* norm_random vs. norm_subjectindipendent
* learn-rate: 1e-3 vs. 1e-4 vs. 1e-5 vs. 1e-6
* batch-size: 64 vs. 32 vs. 128
* Dataset-size(train & test): (1000 & 200) vs. (10000 & 2000) 


In [1]:
# 1: Bib

import time
start_time = time.perf_counter()

import os
import re
import json
import csv
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from torchvision.models import (
    resnet18,
    ResNet18_Weights
)
from datetime import datetime

from torch.utils.tensorboard import SummaryWriter



I0000 00:00:1788774194.903494  373363 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788774197.322732  373363 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:

# 2: Dataset Class: transfer CSV in PyTorch.
class GazeDataset(Dataset):

    def __init__(self, csv_file, transform=None, dataset_size=None, read_all4once=True):

        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.dataset_size = dataset_size if dataset_size is not None else len(self.df)
        self.read_all4once = read_all4once

        if self.read_all4once:
            img = Image.new("RGB", (500, 300))  # any size
            out = transform(img)
            self.images = torch.zeros([self.dataset_size] + list(out.shape))
            self.targets = torch.zeros(self.dataset_size, 2)

        for idx in tqdm(range(self.dataset_size)):

            row = self.df.iloc[idx]

            image = Image.open(
                row["image_name"]
            ).convert("RGB")

            self.targets[idx] = torch.tensor(
                [row["x"], row["y"]],
                dtype=torch.float32
            )

            if self.transform:
                self.images[idx] = self.transform(image)
            else:
                self.images[idx] = image

    def __len__(self):

        return self.dataset_size

    def __getitem__(self, idx):

        return self.images[idx], self.targets[idx]

In [3]:

# 5: func-Diagonal-Error:
def diagonal_errors(model, loader, device):
    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    
    mae = mean_absolute_error(targets, predictions)
    
    rmse = np.sqrt(mean_squared_error(targets, predictions))

    diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    return mae, rmse, diagonal_error_pct

In [4]:
def evaluate_model(model, loader, device):

    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    
    # mae = mean_absolute_error(targets, predictions)
    
    # rmse = np.sqrt(mean_squared_error(targets, predictions))

    # diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    # print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    # print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    # return mae, rmse, diagonal_error_pct, targets, predictions, errors
    return targets, predictions, errors

In [5]:
class GaussianActivation(nn.Module):

    def __init__(self, sigma=1.0):
        super().__init__()
        self.sigma = sigma

    def forward(self, x):
        return torch.exp(
            -(x ** 2) / (2 * self.sigma ** 2)
        )

In [6]:
def sensitive_loss(preds, targets):

    x_total = 0.0

    for i in range(len(targets)):

        x = criterion_base(preds[i], targets[i])

        distance_from_center = (
            torch.abs(targets[i][0] - 0.5) +
            torch.abs(targets[i][1] - 0.5)
        )

        weight = 1.0 + distance_from_center

        x_total += x * weight

    return x_total / len(targets)

# Hypoparameter:

* dataset_size (train-size & test-size)
* batch_size 
* learn-rate
* norm_random vs. norm_subject


In [7]:
# Hypoparameter:

name_dataset_type = ['norm_labels.csv', 'labels.csv']
dataset_size_type = [[10000, 2000], [1000, 200]]
dataset_type = ["norm_subject", "norm_random"]
batch_size_type = [32, 64, 128]
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
epochs_num = [1, 2, 10, 25, 500]



optimizer_name = "AdamW"

# ##########################################################################
# 1.type Dataset: 'norm_labels.csv' or 'labels.csv'
dataset_name = name_dataset_type[0]                       # norm_labels.csv
# dataset_name = name_dataset_type[1]                       # labels.csv
print(f"\n dataset_name: \t {dataset_name}")


# 2.dataset_sizt: (10000 & 2000) or (1000, 200)
def_dataset_size = dataset_size_type[0]                   # [10000, 2000]
# def_dataset_size = dataset_size_type[1]                   # [1000, 200]
print(f"\n def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")


# 3.batch_size: '32', '64' or '128'
# batch_Size = batch_size_type[0]                           # 32
# batch_Size = batch_size_type[1]                           # 64
batch_Size = batch_size_type[2]                           # 128
print(f"\n batch_Size: \t {batch_Size}")


# 4.type of dataset-split: "norm_subject_independed" or "norm_random"
def_dataset = dataset_type[0]                             # norm_subject
# def_dataset = dataset_type[1]                             # norm_random
print(f"\n def_dataset: \t {def_dataset}")


# 5.learning_rate: '1e-3', '1e-4', '1e-5' or '1e-6'
# learning_rate = lr_type[0]                                # 1e-3
# learning_rate = lr_type[1]                                # 1e-4
learning_rate = lr_type[2]                                # 1e-5
# learning_rate = lr_type[3]                                # 1e-6
print(f"\n learning_rate: \t {learning_rate}")


# 6.number of epochs: 1, 2, 10, 25 or 500
epochs = epochs_num[-1]                                   # 500
print(f"\n epochs: \t {epochs}")




weight_Decay = 1e-5

active_func = None

patience = 3               # after 3 Epochen without Optimierung has to stop training
print(f"\n patience: \t {patience}")



 dataset_name: 	 norm_labels.csv

 def_dataset_size: train: 10000, 	 test: 2000

 batch_Size: 	 128

 def_dataset: 	 norm_subject

 learning_rate: 	 1e-05

 epochs: 	 500

 patience: 	 3


In [8]:
# 6: CSV laden
root_folder = "./dataset"
csv_path = Path(f"{root_folder}/{dataset_name}")
# print(f"\n csv_path: {csv_path}")

df = pd.read_csv(csv_path)
df.head()
# check:
# print(df.shape)

,image_name,x,y,subject_ID,screen_w,screen_h
0,dataset/images/00002/00002/frames/00000.jpg,0.500000,0.500000,2,320,568
1,dataset/images/00002/00002/frames/00001.jpg,0.500000,0.500000,2,320,568
2,dataset/images/00002/00002/frames/00002.jpg,0.500000,0.500000,2,320,568
3,dataset/images/00002/00002/frames/00003.jpg,0.500000,0.500000,2,320,568
4,dataset/images/00002/00002/frames/00004.jpg,0.873929,0.114375,2,320,568


In [9]:
# 8: DataLoader
# Transformationen:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1,1.5)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Dataset:


if def_dataset == 'norm_subject':

    train_dataset = GazeDataset(
        "./splits/norm_subject_train.csv",
        transform, dataset_size=def_dataset_size[0],
    )
    
    test_dataset = GazeDataset(
        "./splits/norm_subject_test.csv",
        transform, dataset_size=def_dataset_size[1],
    )



elif def_dataset == 'norm_random':

    train_dataset = GazeDataset(
        "./splits/norm_random_train.csv",
        transform, dataset_size=def_dataset_size[0],
    )
    
    test_dataset = GazeDataset(
        "./splits/norm_random_test.csv",
        transform, dataset_size=def_dataset_size[1],
    )

else:

    raise ValueError(
                "The input as dataset_type is Wrong!"
            )


# Loader:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_Size,
    shuffle=True,
    num_workers=8,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_Size,
    shuffle=False,
    num_workers=8,
    persistent_workers=True
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:18<00:00, 105.81it/s]


In [10]:
# 9: ResNet18 (Pretrainiertes Modell):
# Load:
def reset_model(act=None):
    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model_name = model.__class__.__name__
    
    # ---------------------------------------
    # Activation function for the last layer
    # ---------------------------------------
    if act == "Sigmoid":
        last_layer = nn.Sigmoid()

    elif act == "Gaussian":
        last_layer = GaussianActivation(sigma=1.0)

    elif act == "ReLU":
        last_layer = nn.ReLU()

    elif act == "None":
        last_layer = nn.Identity()

    else:
        raise ValueError(
            f"Unknown activation function: {act}"
        )

    # ---------------------------------------
    # Replace original ResNet FC
    # ---------------------------------------
    model.fc = nn.Sequential(
        nn.Linear(
            model.fc.in_features,
            512
        ),
        nn.ReLU(),

        nn.Linear(
            512,
            256
        ),
        nn.ReLU(),

        nn.Dropout(0.2),

        nn.Linear(
            256,
            128
        ),
        nn.ReLU(),

        nn.Linear(
            128,
            2
        ),

        # Last activation
        last_layer
    )

    return model, model_name

In [11]:
# 10: GPU or CPU
# check:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [12]:
print(f"\n\t dataset_name: \t {dataset_name}")
print(f"\t def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")
print(f"\t batch_Size: \t {batch_Size}")
print(f"\t def_dataset: \t {def_dataset}")
print(f"\t learning_rate: \t {learning_rate}")
print(f"\t epochs: \t {epochs}")
print(f"\t patience: \t {patience}")

for t in range(3):
    
    print(f"\n\t t: {t + 1}")
    
    criterion = nn.L1Loss()
    
    criterion_base = nn.L1Loss()
    
    epochs = epochs
    
    if globals().get("bas_const_err") is None:
        bas_const_err = 27.245
    
    if globals().get("bas_rand_err") is None:
        bas_rand_err = 38.961
        
    
    # model_output = './models'
    # model_dir= Path(model_output)
    
    # if not os.path.exists(model_dir):
    #     model_dir.mkdir(
    #         parents=True,
    #         exist_ok=True
    #     )
    
    
    # diagrams_output = './results/diagrams'
    # diagrams_dir= Path(diagrams_output)
    
    # if not os.path.exists(diagrams_dir):
    #     diagrams_dir.mkdir(
    #         parents=True,
    #         exist_ok=True
    #     )
    
    diag_test_errors = {}
    diag_train_errors = {}
    
    
    act_time_epochs = {}
    
    
    diag_test_error, diag_train_error = [], []
    
    acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']
    
    act = acts[0]
    
    # for act in ['Sigmoid', 'Gaussian', 'ReLU', 'None']:
    # for act in ['Sigmoid']:
    # for act in ['None']:
    
    if act == 'Sigmoid':
    
    # if act == 'None':
    
        train_start = time.perf_counter()
      
        model, model_name = reset_model(act=act)
    
        active_func = act
    
        diag_test_errors[act] = []
        diag_train_errors[act] = []
    
        best_error = float("inf")
    
        t = 0                      # Number Epochen without Optimierung
        e = 0                      # Number Epochen with Optimierung
    
    
        model.to(device)
    
    
    
        for param in model.parameters():
            param.requires_grad = False
    
        # Unfreeze the head
        for param in model.layer4.parameters():
            param.requires_grad=True
    
        for param in model.fc.parameters():
            param.requires_grad=True
    
    
        optimizer = torch.optim.AdamW(
            filter(lambda p:p.requires_grad, model.parameters()),
            lr=learning_rate,
            weight_decay=weight_Decay
        )
    
        print('\n ',"=#=" * 25)
        print(f"\t\t act: {act}")
        print(' ',"=#=" * 25)
    
        print("\n\t Start: \n")
        
        print("train_error:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
        diag_train_error.append(np.round(train_diag_pct, 4))
        
        diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
    
    
        print("\ntest_error:")
        test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
        diag_test_error.append(np.round(test_diag_pct, 4))
    
        diag_test_errors[act].append(np.round(test_diag_pct, 4))

        if best_error is None or test_diag_pct < best_error:    
            # set the first test-error
            best_error = test_diag_pct
            t = 0
            e += 1
    
    
        print("\n\n\t Training: \n")
    
        for epoch in range(epochs):
    
            epoch_start = time.perf_counter()
    
            model.train()
            running_loss = 0
    
            loop = tqdm(
                train_loader,
                desc=f"Epoch {epoch + 1}"
            )
    
            for images, targets in loop:
                images = images.to(device)
                targets = targets.to(device)
                optimizer.zero_grad()
                preds = model(images)
    
                ################################
                
                # Loss 1:
                loss = criterion(
                    preds,
                    targets
                )
    
                
                # Loss 2: 
                # loss = sensitive_loss(
                #     preds,
                #     targets
                # )
                
                ################################
    
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
    
                loop.set_postfix(
                    loss=loss.item()
                )
    
            # epoch_end = time.perf_counter()
    
            print(f"\ntrain_error:")
            train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
            diag_train_error.append(np.round(train_diag_pct, 4))
    
            diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
            print(f"\ntest_error:")
            test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
            diag_test_error.append(np.round(test_diag_pct, 4))
    
            diag_test_errors[act].append(np.round(test_diag_pct, 4))
    
            # --------------------------------------------------
            # Early Stopping
            # --------------------------------------------------
            
            if best_error is None or test_diag_pct < best_error:
            
                # find an improvement
                best_error = test_diag_pct
                t = 0
                e += 1
            
                print(
                    f"Test-Diagonal-Error: {test_diag_pct:.4f}% | "
                    f"Improvment: {e}"
                )
            
                # save the better Modell
                # torch.save(
                #     model.state_dict(),
                #     f"./models/best_optim-model_"
                #     f"{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path"
                # )
    
                # torch.save(
                #     model.state_dict(),
                #     f"./models/best-model_"
                #     f"{model_name}_{act}.path"
                # )
            
            else:
            
                # without imporovement
                t += 1
            
                print(
                    f"Patience: {t}/{patience}"
                )
            
                if t >= patience:
                    print(
                        f"\nEarly Stopping after {epoch + 1} Epochen."
                    )
                    print(
                        f"Imporovments totally: {e}"
                    )
                    break
    
    
    
            epoch_end = time.perf_counter()
    
    
            print(
                f"\n[{datetime.now().strftime('%H:%M:%S')}] Epoch {epoch + 1}: {epoch_end - epoch_start:.2f} Sekunden | "
                f"Running_loss: {running_loss / len(train_loader):.3f} | "
                f"test_diag_error={test_diag_pct:.4f}% | "
                f"train_diag_error={train_diag_pct:.4f}% | \n"
            )
    
            # torch.save(model.state_dict(),
            #         f"./models/last_best_optim_model_{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path")
    
            # torch.save(
            #         model.state_dict(),
            #         f"./models/last-model_"
            #         f"{model_name}_{act}.path"
            #     )
    
    
        train_end = time.perf_counter()
    
    
        elapsed_running_time = train_end - train_start
        print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} Sekunden")
        print(f"totll running-tiems (min): {elapsed_running_time / 60:.2f} Minuten\n")
    
        
        # act_time[act].append(elapsed_running_time)
        # act_epoch[act].append(e)
    
        act_time_epochs[act] = {
            "time_minutes": elapsed_running_time / 60,
            "epochs": epoch + 1,
            "improvements": e,
            "best_test_error": best_error
        }
    
    
    
        # print(f"\n\nDiagonal_train_error: {np.float64(diag_train_error)}")
        # print(f"\n\nDiagonal_test_error: {np.float64(diag_test_error)}")

    for act, values in act_time_epochs.items():
        print(
            f"last_layer: {act}\t"
            f" epochs: {values['epochs']} Epochen\t|"
            f" running_time: {values['time_minutes']:.2f} min\t |"
            f" improvements: {values['improvements']} Epochen\t |"
            f" best_test_error: {values['best_test_error']:.4f} % "
        )

    print('\n ',"=#=" * 25)
    print(' ',"=#=" * 25)



	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 10000, 	 test: 2000
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

	 t: 1

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
		 act: Sigmoid
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=

	 Start: 

train_error:
MAE : 0.2354 	 RMSE: 0.2758 
Diagonal-Error %: 19.5017 %

test_error:
MAE : 0.2350 	 RMSE: 0.2787 
Diagonal-Error %: 19.7072 %


	 Training: 



Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [10:40<00:00,  8.10s/it, loss=0.249]



train_error:
MAE : 0.2339 	 RMSE: 0.2747 
Diagonal-Error %: 19.4212 %

test_error:
MAE : 0.2347 	 RMSE: 0.2790 
Diagonal-Error %: 19.7259 %
Patience: 1/3

[12:15:34] Epoch 1: 1228.79 Sekunden | Running_loss: 0.235 | test_diag_error=19.7259% | train_diag_error=19.4212% | 



Epoch 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [11:40<00:00,  8.86s/it, loss=0.218]



train_error:
MAE : 0.2329 	 RMSE: 0.2736 
Diagonal-Error %: 19.3470 %

test_error:
MAE : 0.2347 	 RMSE: 0.2790 
Diagonal-Error %: 19.7290 %
Patience: 2/3

[12:38:49] Epoch 2: 1394.43 Sekunden | Running_loss: 0.234 | test_diag_error=19.7290% | train_diag_error=19.3470% | 



...


train_error:
MAE : 0.2328 	 RMSE: 0.2734 
Diagonal-Error %: 19.3355 %

test_error:
MAE : 0.2342 	 RMSE: 0.2783 
Diagonal-Error %: 19.6820 %
Patience: 1/3

[15:03:33] Epoch 2: 1205.36 Sekunden | Running_loss: 0.234 | test_diag_error=19.6820% | train_diag_error=19.3355% | 



Epoch 3: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [10:35<00:00,  8.05s/it, loss=0.247]



train_error:
MAE : 0.2317 	 RMSE: 0.2723 
Diagonal-Error %: 19.2523 %

test_error:
MAE : 0.2343 	 RMSE: 0.2785 
Diagonal-Error %: 19.6894 %
Patience: 2/3

[15:23:29] Epoch 3: 1195.58 Sekunden | Running_loss: 0.233 | test_diag_error=19.6894% | train_diag_error=19.2523% | 



Epoch 4: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [10:38<00:00,  8.08s/it, loss=0.227]



train_error:
MAE : 0.2300 	 RMSE: 0.2707 
Diagonal-Error %: 19.1385 %

test_error:
MAE : 0.2346 	 RMSE: 0.2788 
Diagonal-Error %: 19.7110 %
Patience: 3/3

Early Stopping after 4 Epochen.
Imporovments totally: 2

totll running-tiems (s): 5375.58 Sekunden
totll running-tiems (min): 89.59 Minuten

last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 89.59 min	 | improvements: 2 Epochen	 | best_test_error: 19.6820 % 

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=


    dataset_name: 	  norm_labels.csv
    def_dataset_size: train: 1000, 	 test: 200
    batch_Size: 	  128
    dataset-splits:   norm_subject
    learning_rate: 	  0.0001
    epochs: 	      500
    patience: 	      3

## batch_size(32 vs. 64 vs. 132):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       train: 1000, 	 test: 200
 * dataset-splits: 	   norm_subject
 * batch_Size: 	       32 vs. 64 vs. 132      <============
 * learning_rate: 	   1e-4

    dataset_name: 	  norm_labels.csv
    def_dataset_size: train: 1000, 	 test: 200
    dataset-splits:   norm_subject
    learning_rate: 	  0.0001
    epochs: 	      500
    patience: 	      3

# batch_size: 	 32
 
* ** try 1 **:
last_layer: Sigmoid  epochs: 3 Epochen 	|  running_time: 6.97 min 	 |  improvements: 1 Epochen 	 |  best_test_error: 20.0625 % 

* ** try 2 **:
last_layer: Sigmoid  epochs: 3 Epochen 	|  running_time: 6.87 min 	 |  improvements: 1 Epochen 	 |  best_test_error: 20.1573 %  

* ** try 3 **:
last_layer: Sigmoid  epochs: 3 Epochen 	|  running_time: 6.89 min 	 |  improvements: 1 Epochen 	 |  best_test_error: 20.1097 % 

########################################################################
# batch_size: 	 64

* ** try 1 **:
last_layer: Sigmoid  epochs: 4 Epochen 	|  running_time: 9.20 min 	 |  improvements: 2 Epochen 	 |  best_test_error: 20.0352 % 

* ** try 2 **:
last_layer: Sigmoid  epochs: 7 Epochen 	|  running_time: 15.35 min 	 |  improvements: 3 Epochen 	 |  best_test_error: 20.0870 %

* ** try 3 **:
last_layer: Sigmoid  epochs: 4 Epochen 	|  running_time: 9.20 min 	 |  improvements: 2 Epochen 	 |  best_test_error: 20.0832 %

########################################################################
# batch_size: 	 128
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 4 Epochen	|  running_time: 9.05 min	 |  improvements: 2 Epochen	     |  best_test_error: 19.9919 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	|  running_time: 9.21 min	 |  improvements: 2 Epochen	     |  best_test_error: 19.9870 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	|  running_time: 9.20 min	 |  improvements: 2 Epochen	     |  best_test_error: 20.0366 % 


## def_dataset(norm_subject, norm_random):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       train: 1000, 	 test: 200
 * dataset-splits: 	   norm_subject vs. norm_random      <============
 * batch_Size: 	       128
 * learning_rate: 	   1e-4

	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3

# dataset-splits: 	 norm_subject
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.30 min	 | improvements: 2 Epochen	 | best_test_error: 20.1205 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.25 min	 | improvements: 2 Epochen	 | best_test_error: 20.0771 % 

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.22 min	 | improvements: 2 Epochen	 | best_test_error: 20.0507 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.04 min	 | improvements: 2 Epochen	 | best_test_error: 20.0485 % 

* ** try 5 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.21 min	 | improvements: 2 Epochen	 | best_test_error: 20.0101 % 

* ** try 6 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 11.28 min	 | improvements: 3 Epochen	 | best_test_error: 19.9546 % 

########################################################################
# dataset-splits: 	 norm_random

* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.24 min	 | improvements: 1 Epochen	 | best_test_error: 18.8797 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.14 min	 | improvements: 2 Epochen	 | best_test_error: 18.8308 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.24 min	 | improvements: 2 Epochen	 | best_test_error: 18.8890 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.98 min	 | improvements: 2 Epochen	 | best_test_error: 18.8544 %

* ** try 5 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.72 min	 | improvements: 2 Epochen	 | best_test_error: 18.9156 %

* ** try 6 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.17 min	 | improvements: 2 Epochen	 | best_test_error: 18.8534 %

## learning-rate(1e-3, 1e-4, 1e-5, 1e-6):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       train: 1000, 	 test: 200
 * dataset-splits: 	   norm_subject
 * batch_Size: 	       32
 * learning_rate: 	   0.001 vs. 0.0001 vs. 0.00001 vs. 0.000001    <============

	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.001
	 epochs: 	 500
	 patience: 	 3

# learning_rate:	 1e-3
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.98 min	 | improvements: 1 Epochen	 | best_test_error: 20.1148 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.97 min	 | improvements: 1 Epochen	 | best_test_error: 19.9747 % 

* ** try 3 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.88 min	 | improvements: 1 Epochen	 | best_test_error: 20.1573 % 

########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-04
	 epochs: 	 500
	 patience: 	 3
     
# learning_rate:	 1e-4

* ** try 1 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.85 min	 | improvements: 2 Epochen	 | best_test_error: 20.0753 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.02 min	 | improvements: 2 Epochen	 | best_test_error: 20.1136 % 

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.02 min	 | improvements: 2 Epochen	 | best_test_error: 20.0532 % 

########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3
     
# learning_rate:	 1e-5
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.76 min	 | improvements: 1 Epochen	 | best_test_error: 20.0623 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.13 min	 | improvements: 1 Epochen	 | best_test_error: 20.0904 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 8 Epochen	| running_time: 17.18 min	 | improvements: 5 Epochen	 | best_test_error: 20.1262 %

########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-06
	 epochs: 	 500
	 patience: 	 3
     
# learning_rate:	 1e-6
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 38 Epochen	| running_time: 80.17 min	 | improvements: 30 Epochen	 | best_test_error: 20.1475 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 11.57 min	 | improvements: 3 Epochen	 | best_test_error: 20.1175 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.40 min	 | improvements: 1 Epochen	 | best_test_error: 20.2281 %


## dataset_size:((10000, 2000) vs. (1000, 200)):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       (10000, 2000) vs. (1000, 200)    <============  
 * dataset-splits: 	   norm_subject
 * batch_Size: 	       32
 * learning_rate: 	   1e-4   

	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 10000, 	 test: 2000
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

# dataset_size:       (10000, 2000)
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 77.32 min	 | improvements: 1 Epochen	 | best_test_error: 19.7072 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 70.89 min	 | improvements: 1 Epochen	 | best_test_error: 19.6712 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 89.59 min	 | improvements: 2 Epochen	 | best_test_error: 19.6820 %
########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

# dataset_size:       (1000, 200)

* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.76 min	 | improvements: 1 Epochen	 | best_test_error: 20.0623 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.13 min	 | improvements: 1 Epochen	 | best_test_error: 20.0904 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 8 Epochen	| running_time: 17.18 min	 | improvements: 5 Epochen	 | best_test_error: 20.1262 %


In [ ]:
 dataset_name: 	 norm_labels.csv

 def_dataset_size: train: 1000, 	 test: 200

 batch_Size: 	 32

 def_dataset: 	 norm_subject

 learning_rate: 	 0.0001